<a href="https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
# ============================================================
# W03 — DuckDB + Hugging Face Warehouse Setup
# ============================================================

!pip -q install duckdb

import duckdb
from google.colab import userdata

# ------------------------------------------------------------
# 1. Create DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

# ------------------------------------------------------------
# 2. Get Hugging Face token from Colab Secrets
# ------------------------------------------------------------

hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found.\n\n"
        "Go to Colab → Secrets → Add a new secret:\n"
        "Name: HF_TOKEN\n"
        "Value: your Hugging Face READ token"
    )

# ------------------------------------------------------------
# 3. Store token securely in DuckDB
# ------------------------------------------------------------

con.execute(
    "SET VARIABLE hf_token = ?",
    [hf_token]
)

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# ------------------------------------------------------------
# 4. Define warehouse paths
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Feature window: February 2026
FEB = f"{FACT}/month=2026-02/*.parquet"

# Label window: March 2026
MAR = f"{FACT}/month=2026-03/*.parquet"

print("✓ DuckDB connected")
print("✓ Hugging Face authentication configured")
print("✓ Warehouse paths configured")
print()
print("Feature window:", "February 2026")
print("Label window:", "March 2026")

✓ DuckDB connected
✓ Hugging Face authentication configured
✓ Warehouse paths configured

Feature window: February 2026
Label window: March 2026


## 1. My rule and its reason codes

### My rule

I will rank content items using two observable signals from the February 2026 feature window:

1. **GSC impressions** — represents the amount of measured search visibility/demand.
2. **GSC CTR** — calculated as total GSC clicks divided by total GSC impressions.

The baseline gives the highest priority to content items that have **high search demand but zero/very low CTR**. These pages have substantial measured visibility but are capturing relatively few clicks, making them candidates for human review.

The thresholds are determined from the February feature-window distribution rather than from the March outcome window.

### Reason codes

- `HIGH_DEMAND_LOW_CTR` — high impressions combined with low CTR; highest-priority review condition.
- `HIGH_DEMAND` — high impressions but CTR does not meet the low-CTR condition.
- `LOW_CTR` — low CTR without sufficiently high search demand.
- `NO_STRONG_SIGNAL` — neither signal meets the baseline threshold.

### Actions

- `REVIEW` — prioritize the content item for human investigation.
- `MONITOR` — retain it at lower priority without claiming that a change is required.

The baseline is a prioritization rule, not a causal claim. A high score indicates that a page matches the observed signal pattern; it does not prove that refreshing the page will improve its future performance.

In [1]:
# Candidate signals for the baseline rule.
# Thresholds will be chosen only after the signal checks below.

rule_signals = {
    "demand": "gsc_impressions",
    "click_performance": "gsc_ctr",
    "visibility_context": "gsc_avg_position"
}

reason_codes = [
    "HIGH_DEMAND_LOW_CTR",
    "VISIBLE_LOW_CTR",
    "HIGH_DEMAND",
    "NO_STRONG_SIGNAL"
]

action_labels = [
    "REVIEW",
    "MONITOR"
]

print("Candidate baseline signals:")
for role, signal in rule_signals.items():
    print(f"  {role}: {signal}")

print("\nPossible reason codes:")
for code in reason_codes:
    print(f"  {code}")

print("\nPossible actions:")
for action in action_labels:
    print(f"  {action}")

print(
    "\nThresholds are intentionally not fixed yet; "
    "the signal checks below will determine whether the proposed rule is supported."
)

Candidate baseline signals:
  demand: gsc_impressions
  click_performance: gsc_ctr
  visibility_context: gsc_avg_position

Possible reason codes:
  HIGH_DEMAND_LOW_CTR
  VISIBLE_LOW_CTR
  HIGH_DEMAND
  NO_STRONG_SIGNAL

Possible actions:
  REVIEW
  MONITOR

Thresholds are intentionally not fixed yet; the signal checks below will determine whether the proposed rule is supported.


## 2. Build the ranked queue (writes the CSV)

### Baseline scoring and ranked queue

The February 2026 daily observations are aggregated to one row per `client_hash_id × content_hash_id`.

For each content item, I calculate total February GSC impressions, total February GSC clicks, and aggregate CTR:

`CTR = total clicks / total impressions`

The baseline assigns one point for high impressions and one point for low CTR. Therefore:

- **Score 2** = high demand + low CTR → `REVIEW`
- **Score 1** = one supporting signal → `MONITOR`
- **Score 0** = neither supporting signal → `MONITOR`

The thresholds are based on the February feature-window distribution. March data and future outcome fields are not used to construct the score.

The resulting ranking is written to `work/outputs/baseline_action_score.csv`.

In [4]:
# ============================================================
# Section 2 — Build the ranked baseline queue
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 1. Aggregate February data to client × content grain
# ------------------------------------------------------------

feb_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions)
            ELSE NULL
        END AS ctr,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_days

    FROM read_parquet('{FEB}')

    WHERE gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print("February feature rows:", len(feb_features))

display(feb_features.head())

# ------------------------------------------------------------
# 2. Keep content items with measurable GSC data
# ------------------------------------------------------------

baseline = feb_features[
    (feb_features["gsc_available_days"] > 0) &
    (feb_features["impressions"] > 0)
].copy()

print("Content items with measurable GSC impressions:", len(baseline))

# ------------------------------------------------------------
# 3. Define transparent baseline thresholds
# ------------------------------------------------------------

high_demand_threshold = baseline["impressions"].quantile(0.75)
low_ctr_threshold = baseline["ctr"].quantile(0.25)

print(f"High-demand threshold (75th percentile): {high_demand_threshold:,.0f} impressions")
print(f"Low-CTR threshold (25th percentile): {low_ctr_threshold:.4%}")

# ------------------------------------------------------------
# 4. Apply ONE transparent baseline rule
#
# High demand + low CTR = highest review priority
# ------------------------------------------------------------

baseline["high_demand"] = (
    baseline["impressions"] >= high_demand_threshold
)

baseline["low_ctr"] = (
    baseline["ctr"] <= low_ctr_threshold
)

# Score:
# 2 = high demand + low CTR
# 1 = one of the two signals
# 0 = neither
baseline["score"] = (
    baseline["high_demand"].astype(int)
    + baseline["low_ctr"].astype(int)
)

# ------------------------------------------------------------
# 5. Reason code
# ------------------------------------------------------------

baseline["reason_code"] = "NO_STRONG_SIGNAL"

baseline.loc[
    baseline["high_demand"] & baseline["low_ctr"],
    "reason_code"
] = "HIGH_DEMAND_LOW_CTR"

baseline.loc[
    baseline["high_demand"] & ~baseline["low_ctr"],
    "reason_code"
] = "HIGH_DEMAND"

baseline.loc[
    ~baseline["high_demand"] & baseline["low_ctr"],
    "reason_code"
] = "VISIBLE_LOW_CTR"

# ------------------------------------------------------------
# 6. Action label
# ------------------------------------------------------------

baseline["action"] = "MONITOR"

baseline.loc[
    baseline["score"] == 2,
    "action"
] = "REVIEW"

# ------------------------------------------------------------
# 7. Rank the queue
# ------------------------------------------------------------

baseline = baseline.sort_values(
    by=["score", "impressions", "ctr"],
    ascending=[False, False, True]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# ------------------------------------------------------------
# 8. Select final output columns
# ------------------------------------------------------------

queue = baseline[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions",
        "clicks",
        "ctr",
        "score",
        "reason_code",
        "action"
    ]
].copy()

# ------------------------------------------------------------
# 9. Write required CSV
# ------------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(
    output_path,
    index=False
)

print(f"✓ Ranked queue created: {len(queue):,} rows")
print(f"✓ CSV written to: {output_path}")

print("\nTop 10:")
display(queue.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February feature rows: 153559


,client_hash_id,content_hash_id,impressions,clicks,ctr,gsc_available_days
0,client_e547b89c05043229,content_7995404695ee1ffd,1012.0,3.0,0.002964,28
1,client_e547b89c05043229,content_ccbb253f142217c3,1598.0,7.0,0.004380,28
2,client_e547b89c05043229,content_ae16a6b9cf64c80a,861.0,0.0,0.000000,28
3,client_e547b89c05043229,content_acf700633f016e5a,245.0,0.0,0.000000,28
4,client_e547b89c05043229,content_712e44562fed8ff2,306.0,0.0,0.000000,28


Content items with measurable GSC impressions: 153559
High-demand threshold (75th percentile): 766 impressions
Low-CTR threshold (25th percentile): 0.0000%
✓ Ranked queue created: 153,559 rows
✓ CSV written to: work/outputs/baseline_action_score.csv

Top 10:


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,score,reason_code,action
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
2,3,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
3,4,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
4,5,client_23a62021009f63c4,content_1162dc8495e06dfb,19938.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
5,6,client_23a62021009f63c4,content_c367b0ca57f3559b,19627.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
6,7,client_23a62021009f63c4,content_67a19b4e8f52924e,19529.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
7,8,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,18472.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
8,9,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,15238.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW
9,10,client_23a62021009f63c4,content_73aa61dcedebbf30,15050.0,0.0,0.0,2,HIGH_DEMAND_LOW_CTR,REVIEW


## 3. Top-20 review

The top 20 content items are reviewed as a sanity check on the baseline ranking.

For each item, I record:

- **Action** — the action assigned by the baseline.
- **Reason code** — the signal combination responsible for the score.
- **Confidence note** — confidence that the item matches the baseline rule, not confidence that the recommended action will cause improvement.
- **What would make it wrong** — a plausible explanation for why the observed signal pattern may not represent a genuine content opportunity.

In this run, the strongest-ranked items are expected to be dominated by the `HIGH_DEMAND_LOW_CTR` reason code because the highest score requires both high impressions and low CTR.

The review is observational and does not establish that refreshing or changing a page will cause its performance to improve.

In [5]:
# ============================================================
# Section 3 — Top-20 review
# ============================================================

# Take the 20 highest-ranked items from the baseline queue
top20 = queue.head(20).copy()

# ------------------------------------------------------------
# Confidence note
# ------------------------------------------------------------

def confidence_note(row):
    if row["score"] == 2:
        return "High within baseline: both demand and CTR signals are present."
    elif row["score"] == 1:
        return "Medium within baseline: only one supporting signal is present."
    else:
        return "Low within baseline: no strong signal is present."


top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

# ------------------------------------------------------------
# What could make the recommendation wrong?
# ------------------------------------------------------------

def wrong_if(row):
    if row["reason_code"] == "HIGH_DEMAND_LOW_CTR":
        return (
            "Low CTR may be explained by query intent, search position, "
            "or measurement limitations rather than a content opportunity."
        )

    elif row["reason_code"] == "HIGH_DEMAND":
        return (
            "High impressions do not necessarily indicate a problem; "
            "the page may already be performing appropriately for its queries."
        )

    elif row["reason_code"] == "VISIBLE_LOW_CTR":
        return (
            "Low CTR may be expected for the page's query intent or position, "
            "so the signal alone may not justify a review."
        )

    else:
        return (
            "The baseline may miss relevant context that is not represented "
            "by these signals."
        )


top20["what_would_make_it_wrong"] = top20.apply(
    wrong_if,
    axis=1
)

# ------------------------------------------------------------
# Display the top 20 review
# ------------------------------------------------------------

top20_review = top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

print("TOP-20 BASELINE REVIEW")
print("=" * 80)

display(top20_review)


TOP-20 BASELINE REVIEW


,rank,client_hash_id,content_hash_id,score,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,client_73cda7b4e4f265ea,content_fec55986a1868d62,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
1,2,client_73cda7b4e4f265ea,content_c9f840183215651b,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
2,3,client_23a62021009f63c4,content_2f09787bdf392b16,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
3,4,client_23a62021009f63c4,content_559cdd76da9306de,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
4,5,client_23a62021009f63c4,content_1162dc8495e06dfb,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
5,6,client_23a62021009f63c4,content_c367b0ca57f3559b,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
6,7,client_23a62021009f63c4,content_67a19b4e8f52924e,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
7,8,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
8,9,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."
9,10,client_23a62021009f63c4,content_73aa61dcedebbf30,2,REVIEW,HIGH_DEMAND_LOW_CTR,High within baseline: both demand and CTR sign...,"Low CTR may be explained by query intent, sear..."


## 4. Weak picks + leakage check

### Weak picks + leakage check

I review weaker picks to identify cases where the simple rule may produce false positives. A weak pick receives only one supporting signal rather than the strongest combination of high demand and low CTR.

The baseline must use only information available during the February 2026 feature window. It must not use March metrics, future-window information, label-derived fields, or FlyRank product decision outputs such as `priority_score`, `action_type`, or `health_score`.

The leakage check confirms that the baseline is based on observable February search signals rather than reproducing an existing product decision or using future information.

A weak or incorrect recommendation is an expected possibility for a simple baseline; identifying these limitations is part of evaluating the rule.

In [6]:
# ============================================================
# Section 4 — Weak picks + leakage check
# ============================================================

print("=" * 80)
print("1. WEAK PICKS")
print("=" * 80)

# Weak picks = score 1.
# These have only one of the two main supporting signals.
weak_picks = queue[queue["score"] == 1].copy()

display(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "impressions",
            "clicks",
            "ctr",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

print(f"Number of score-1 rows: {len(weak_picks):,}")


# ------------------------------------------------------------
# 2. Leakage check — forbidden fields
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("2. LEAKAGE CHECK — FORBIDDEN FIELDS")
print("=" * 80)

forbidden_fields = {
    # Future-window / label-derived fields
    "went_dark",
    "march_clicks",
    "march_impressions",
    "march_ctr",

    # Product-derived decision fields
    "priority_score",
    "action_type",
    "health_score",
    "needs_ctr_fix",
    "is_quick_win"
}

used_fields = set(baseline.columns)

forbidden_used = used_fields.intersection(forbidden_fields)

print("Forbidden fields found in baseline:", forbidden_used)

assert not forbidden_used, (
    f"Potential leakage/product decision fields found: {forbidden_used}"
)

print("✓ No forbidden product-decision or label-derived fields found.")


# ------------------------------------------------------------
# 3. Confirm the baseline is based on February signals
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("3. FEATURE WINDOW CHECK")
print("=" * 80)

print("Baseline source:")
print("  → February 2026 feature window")
print("  → GSC impressions")
print("  → GSC clicks")
print("  → CTR derived from February clicks / impressions")
print()
print("Future label window:")
print("  → March 2026")
print("  → NOT used by the baseline")

print("✓ Baseline uses pre-decision information only.")


# ------------------------------------------------------------
# 4. Check the output columns
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("4. OUTPUT CHECK")
print("=" * 80)

expected_output_columns = {
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "score",
    "reason_code",
    "action"
}

actual_output_columns = set(queue.columns)

missing_output_columns = expected_output_columns - actual_output_columns

print("Missing required output fields:", missing_output_columns)

assert not missing_output_columns, (
    f"Missing required queue fields: {missing_output_columns}"
)

print("✓ Ranked queue contains the required output fields.")


# ------------------------------------------------------------
# 5. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("FINAL LEAKAGE / WEAK-PICK SUMMARY")
print("=" * 80)

print(f"Total ranked rows: {len(queue):,}")
print(f"Score-2 rows: {(queue['score'] == 2).sum():,}")
print(f"Score-1 rows: {(queue['score'] == 1).sum():,}")
print(f"Score-0 rows: {(queue['score'] == 0).sum():,}")

print("\n✓ Weak-pick review completed.")
print("✓ Leakage check passed.")

1. WEAK PICKS


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,score,reason_code,action
4250,4251,client_73cda7b4e4f265ea,content_8e1334d6356668e3,203401.0,2.0,0.000010,1,HIGH_DEMAND,MONITOR
4251,4252,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,195648.0,1.0,0.000005,1,HIGH_DEMAND,MONITOR
4252,4253,client_73cda7b4e4f265ea,content_512dbad65bd5ade9,167303.0,3310.0,0.019784,1,HIGH_DEMAND,MONITOR
4253,4254,client_73cda7b4e4f265ea,content_e241d6415ac9e534,164152.0,401.0,0.002443,1,HIGH_DEMAND,MONITOR
4254,4255,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,627.0,0.003867,1,HIGH_DEMAND,MONITOR
4255,4256,client_62f4a7e64f5e0096,content_b99ea6861864dea5,160699.0,273.0,0.001699,1,HIGH_DEMAND,MONITOR
4256,4257,client_62f4a7e64f5e0096,content_f107e54b10b43725,156163.0,883.0,0.005654,1,HIGH_DEMAND,MONITOR
4257,4258,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,154502.0,2508.0,0.016233,1,HIGH_DEMAND,MONITOR
4258,4259,client_62f4a7e64f5e0096,content_acbcc847f8996314,148256.0,239.0,0.001612,1,HIGH_DEMAND,MONITOR
4259,4260,client_e547b89c05043229,content_c9a0c2fdbdbfb562,142215.0,1605.0,0.011286,1,HIGH_DEMAND,MONITOR


Number of score-1 rows: 128,369

2. LEAKAGE CHECK — FORBIDDEN FIELDS
Forbidden fields found in baseline: set()
✓ No forbidden product-decision or label-derived fields found.

3. FEATURE WINDOW CHECK
Baseline source:
  → February 2026 feature window
  → GSC impressions
  → GSC clicks
  → CTR derived from February clicks / impressions

Future label window:
  → March 2026
  → NOT used by the baseline
✓ Baseline uses pre-decision information only.

4. OUTPUT CHECK
Missing required output fields: set()
✓ Ranked queue contains the required output fields.

FINAL LEAKAGE / WEAK-PICK SUMMARY
Total ranked rows: 153,559
Score-2 rows: 4,250
Score-1 rows: 128,369
Score-0 rows: 20,940

✓ Weak-pick review completed.
✓ Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.